# Reading the vault as a whole

This notebook shows the vault as a coder would read it: the big picture first, then one concern at a time down to the technologies behind it. It only reads the vault; it writes nothing into it, except edge tables into `exports/`, which git ignores.

Run it with **Run > Run All Cells**. Every claim made here is checked by an `assert`, so a cell that stops with an `AssertionError` tells you the picture and the pages disagree.

## Load the vault

The tool reads the pages and checks them. If it finds an error, the notebook lists the problems and shows nothing else, because a picture of a vault with errors would mislead.

In [ ]:
import csv
from collections import deque
from pathlib import Path

from IPython.display import Markdown, display

import scat_kg
from scat_kg import draw

LAYERS = ("group", "concern", "affordance", "technology")  # "up" is toward groups


def table(headers, rows):
    """Show rows as a plain table."""
    cell = lambda x: str(x).replace("|", "\\|")
    lines = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    lines += ["| " + " | ".join(cell(x) for x in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


def edge_rows(edges):
    return [(e["type"], e["from"], e["to"], "; ".join(e["narratives"]), e["status"]) for e in edges]


EDGE_HEADERS = ["type", "from", "to", "told by", "status"]

report = scat_kg.check()
errors = [p for p in report.problems if p in report.errors]
READY = not errors
if READY:
    graph = scat_kg.load()
    print(f"The vault has no errors: {len(graph.narratives)} narratives, "
          f"{sum(len(graph.nodes[layer]) for layer in LAYERS)} nodes, {len(graph.edges)} edges.")
    if report.warnings:
        print(f"({len(report.warnings)} warnings: run `uv run scat check` to read them.)")
else:
    print(f"The vault has {len(errors)} error(s). Fix them, then run the notebook again.\n")
    print(report)
SKIP = "Skipped: the vault has errors (see the cell that loads it)."

## Fingerprint view: what does each narrative tell?

The fingerprint of a narrative is the nodes it links and the edges its table tells. It answers: what does this one account say, and is any of it left unsaid? A candidate is a pair the narrative links but has no row for.

In [ ]:
fingerprints = []
if READY:
    view = scat_kg.fingerprint()
    if not view.ok:
        print(view)
    else:
        fingerprints = [n for n in view.to_dict()["narratives"] if n["nodes"]]
        for n in fingerprints:
            display(Markdown(f"### {n['name']} ({n['kind']}, {n['status']})"))
            display(draw.fingerprint(n["name"]))
            table(["layer", "node", "status"], [(x["type"], x["name"], x["status"]) for x in n["nodes"]])
            table(EDGE_HEADERS, edge_rows(n["edges"]) or [("none", "", "", "", "")])
            if n["candidates"]:
                display(Markdown("Candidates: pairs this narrative links but tells no row for."))
                table(["candidate type", "from", "to"], [(c["type"], c["from"], c["to"]) for c in n["candidates"]])
else:
    print(SKIP)

Each drawing has the four layers as columns, groups on the left and technologies on the right. A line is an edge. A dashed line is a candidate: the narrative links both ends but tells no row for it. Relationship annotations never appear.

In [ ]:
if READY:
    for n in fingerprints:
        told = next(x for x in graph.narratives if x.name == n["name"])
        assert [x["name"] for x in n["nodes"]] == list(told.nodes), n["name"]
        assert [(e["type"], e["from"], e["to"]) for e in n["edges"]] == [
            (k.type, k.from_node, k.to_node) for k in told.edges
        ], n["name"]
        # Only the four layers: no narrative and no relationship annotation is drawn as a node.
        assert {x["type"] for x in n["nodes"]} <= set(LAYERS), n["name"]
        assert set(n) == {"name", "kind", "status", "nodes", "edges", "candidates"}, n["name"]
    print(f"Checked {len(fingerprints)} fingerprints against the loader: the same nodes, the same edges.")
else:
    print(SKIP)

## Edge tables: who holds what, and what answers it?

An edge table lists every edge of one type, between two neighbouring layers. It answers: what are all the `holds`, `addressed-by` and `provided-by` edges, and which narratives tell each? The same tables are written to `exports/` as CSV files you can open in a spreadsheet.

In [ ]:
EDGE_TYPES = ("holds", "addressed-by", "provided-by")
tables = {}
if READY:
    for slug in EDGE_TYPES:
        v = scat_kg.edge_table(slug)
        if not v.ok:
            print(v)
            continue
        tables[slug] = v.to_dict()
        display(Markdown(f"### {slug}: {tables[slug]['from_layer']} to {tables[slug]['to_layer']}"))
        table(["from", "to", "told by", "status"],
              [(e["from"], e["to"], "; ".join(e["narratives"]), e["status"]) for e in tables[slug]["rows"]])
    result = scat_kg.export_edges("exports")
    print(result)
else:
    print(SKIP)

In [ ]:
if READY:
    for slug, doc in tables.items():
        path = Path("exports") / f"{slug}.csv"
        assert path.exists(), path
        with path.open(newline="", encoding="utf-8") as f:
            written = list(csv.DictReader(f))
        shown = [
            {"from": e["from"], "to": e["to"], "narratives": ";".join(e["narratives"]), "status": e["status"]}
            for e in doc["rows"]
        ]
        assert written == shown, slug
    print("The three CSV files exist and hold exactly the rows shown above.")
else:
    print(SKIP)

## Layered view: what is the whole picture?

The layered view joins the fingerprints of every narrative into one drawing. It answers: across everything the narratives tell, which groups hold which concerns, and which affordances and technologies stand behind them?

In [ ]:
if READY:
    display(draw.layered())
else:
    print(SKIP)

Read it from left to right: a group, the concerns it holds, the affordances that address them, the technologies that provide them. A node with many lines is shared by many accounts. A node with few lines may be a gap worth a second look in the field notes.

In [ ]:
if READY:
    whole = scat_kg.layered().to_dict()
    assert whole["narratives"] == [n.name for n in graph.narratives]
    assert list(whole["nodes"]) == list(LAYERS)
    assert {e["type"] + "|" + e["from"] + "|" + e["to"] for e in whole["edges"]} == {
        f"{e.type}|{e.from_node}|{e.to_node}" for e in graph.edges
    }
    print(f"The layered view holds all {len(graph.edges)} edges of the vault.")
else:
    print(SKIP)

## Traces: which technologies sit behind a concern?

A trace follows edges from one node, up toward groups and down toward technologies. It answers: who cares about this, and what delivers it? Below, one node is picked from each layer, the one with the most edges.

In [ ]:
def reach(start, up):
    """The nodes reached from `start`, found by walking the exported CSV files alone (breadth first)."""
    pairs = []
    for slug in EDGE_TYPES:
        with (Path("exports") / f"{slug}.csv").open(newline="", encoding="utf-8") as f:
            pairs += [(r["from"], r["to"]) for r in csv.DictReader(f)]
    near, far = (1, 0) if up else (0, 1)
    seen, todo = set(), deque([start])
    while todo:
        here = todo.popleft()
        for pair in pairs:
            if pair[near] == here and pair[far] not in seen:
                seen.add(pair[far])
                todo.append(pair[far])
    return seen


if READY:
    degree = {}
    for e in graph.edges:
        for name in (e.from_node, e.to_node):
            degree[name] = degree.get(name, 0) + 1
    chosen = []
    for layer in LAYERS:
        linked = sorted(n.name for n in graph.nodes[layer] if n.name in degree)
        if linked:
            chosen.append((layer, max(linked, key=lambda name: degree[name])))
    for layer, name in chosen:
        t = scat_kg.trace(name)
        if not t.ok:
            print(t)
            continue
        d = t.to_dict()
        display(Markdown(f"### {name} ({layer})"))
        rows = [("up", x["type"], x["name"]) for x in d["up"]] + [("down", x["type"], x["name"]) for x in d["down"]]
        table(["direction", "layer", "node"], rows or [("none", "", "")])
        assert {x["name"] for x in d["up"]} == reach(name, up=True), (name, "up")
        assert {x["name"] for x in d["down"]} == reach(name, up=False), (name, "down")
    print(f"Checked {len(chosen)} traces against a walk over the exported CSV files: the same nodes up and down.")
else:
    print(SKIP)

A trace leaves out the node it starts from. Up and down are separate walks: a concern's groups are not mixed with its technologies, and a node is not reached by going down and then back up.

## Where to go next

Every view here has a twin on the command line, `scat fingerprint`, `scat edges`, `scat layered`, `scat trace`, `scat export-edges` and `scat draw`, and the same names work in the Python REPL (`scat_kg.trace("a-node")`). The guides on the docs site, <https://dynamicalsystemsgroup.github.io/scat-kg-tools/>, say how to read the views and take them back to the field notes.